# 🏛️ Gemma 4 E2B — QLoRA Fine-Tuning for Regulatory Obligation Extraction

**Pipeline:** Unsloth + MLflow + PEFT (QLoRA / 4-bit NF4)  
**Target Model:** `google/gemma-4-2b-it` (4-bit quantized, QLoRA adapters)  
**Task:** Obligation Extraction — Span-Level F1 + Modality Classification (MUST vs SHOULD)  
**GPU:** Google Colab T4 (16 GB VRAM)

---
### Key Difference from LoRA Notebook
| Feature | LoRA | **QLoRA (this notebook)** |
|---|---|---|
| Base precision | BF16 | **4-bit NF4** |
| VRAM usage | ~10–12 GB | **~6–8 GB** |
| Training speed | Faster | Slightly slower (dequant overhead) |
| Adapter quality | High | Comparable (double quant + NF4) |
| Best for | ≥ 24 GB VRAM | **T4 / 16 GB VRAM** |

### Pipeline Phases
1. **Data Ingestion & Validation** — Schema enforcement, 80/10/10 split
2. **Fine-Tuning** — QLoRA via Unsloth + MLflow tracking
3. **Evaluation & Benchmarking** — Span-F1, Modality Accuracy, LegalBench, ObliQA


## 📦 Phase 0 — Install Dependencies

In [ ]:
pip uninstall -y unsloth unsloth_zoo trl peft accelerate bitsandbytes xformers torch

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128


In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Install Unsloth (Colab-optimised)
#!pip install --force-reinstall "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes xformers -q

# MLflow + evaluation libraries
!pip install mlflow datasets evaluate seqeval rouge_score bert_score -q
!pip install jsonschema scikit-learn pandas numpy -q

print('✅ All dependencies installed.')

✅ All dependencies installed.


In [3]:
import os, json, re, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

from unsloth import FastLanguageModel
import torch
from datasets import Dataset, DatasetDict, load_dataset
from transformers import TrainingArguments, TrainerCallback, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

import mlflow
import mlflow.pytorch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import evaluate

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f'🔥 PyTorch: {torch.__version__}')
print(f'🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🔥 PyTorch: 2.10.0+cu128
🖥️  GPU: Tesla T4
💾 VRAM: 15.6 GB


## ⚙️ Phase 0.1 — Global Configuration (QLoRA-Specific)

In [4]:
# Redefining Config as a standard dictionary to ensure picklability
cfg = {
    "model_name": "google/gemma-4-E2B-it",
    "max_seq_length": 1024,
    "load_in_4bit": True,         # Enabled for QLoRA
    "dtype": torch.float16,       # FP16 for T4
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 5,
    "weight_decay": 0.01,
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 1.0,
    "logging_steps": 10,
    "eval_steps": 50,
    "save_steps": 100,
    "output_dir": "./gemma4_qlora_regulatory",
    "run_name": "qlora-r16-lr2e4-ep5",
    "mlflow_experiment": "gemma4_qlora-regulatory-obligations",
    "data_path": "resampled_ft-v2.jsonl"
}

os.makedirs(cfg["output_dir"], exist_ok=True)
print('✅ Config redo complete.')
# Fix: Use .items() instead of .__dict__.items() for standard dicts
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

✅ Config redo complete.
{
  "model_name": "google/gemma-4-E2B-it",
  "max_seq_length": "1024",
  "load_in_4bit": "True",
  "dtype": "torch.float16",
  "lora_r": "16",
  "lora_alpha": "16",
  "lora_dropout": "0.05",
  "lora_target_modules": "['q_proj', 'k_proj', 'v_proj', 'o_proj']",
  "learning_rate": "0.0001",
  "per_device_train_batch_size": "1",
  "gradient_accumulation_steps": "4",
  "num_train_epochs": "5",
  "weight_decay": "0.01",
  "warmup_steps": "10",
  "lr_scheduler_type": "cosine",
  "max_grad_norm": "1.0",
  "logging_steps": "10",
  "eval_steps": "50",
  "save_steps": "100",
  "output_dir": "./gemma4_qlora_regulatory",
  "run_name": "qlora-r16-lr2e4-ep5",
  "mlflow_experiment": "gemma4_qlora-regulatory-obligations",
  "data_path": "resampled_ft-v2.jsonl"
}


## 📐 Phase 1 — Data Schema, Ingestion & Validation

In [5]:
from jsonschema import validate, ValidationError

# Updated Schema: Added MUST_NOT, SHOULD_NOT, and ASPIRATIONAL_OBLIGATION
OBLIGATION_SCHEMA = {
    "type": "object",
    "required": ["metadata", "context", "output"],
    "properties": {
        "metadata": {
            "type": "object",
            "required": ["source_id", "regulation_name", "section"]
        },
        "instruction": {"type": ["string", "null"]},
        "context": {"type": "string"},
        "tool_use": {
            "oneOf": [
                {"type": "null"},
                {
                    "type": "object",
                    "properties": {
                        "available_tools": {"type": "array", "items": {"type": "string"}},
                        "tool_rationale": {"type": ["string", "null"]},
                        "tool_call": {"type": ["object", "null"]},
                        "external_context": {"type": ["string", "null"]}
                    }
                }
            ]
        },
        "thought_trace": {"type": ["string", "null"]},
        "output": {
            "oneOf": [
                {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "required": ["obligation_id", "subject", "modality", "action", "reference_anchor"],
                        "properties": {
                            "obligation_id": {"type": "string"},
                            "subject": {"type": "string"},
                            "modality": {"type": "string", "enum": ["MUST", "SHOULD", "MAY", "PROHIBITED", "MUST_NOT", "SHOULD_NOT", "ASPIRATIONAL_OBLIGATION"]},
                            "action": {"type": "string"},
                            "conditions": {"type": "string"},
                            "deadline": {"type": "string"},
                            "reference_anchor": {"type": "string"}
                        }
                    }
                },
                {
                    "type": "object",
                    "required": ["status", "message", "reason"],
                    "properties": {
                        "status": {"type": "string", "enum": ["rejected", "no_obligation"]},
                        "message": {"type": "string"},
                        "reason": {"type": "string"}
                    }
                }
            ]
        }
    }
}

def validate_record(record: dict) -> Tuple[bool, str]:
    try:
        validate(instance=record, schema=OBLIGATION_SCHEMA)
        return True, ""
    except ValidationError as e:
        return False, e.message

print('✅ Updated Schema to include MUST_NOT, SHOULD_NOT, and ASPIRATIONAL_OBLIGATION.')

✅ Updated Schema to include MUST_NOT, SHOULD_NOT, and ASPIRATIONAL_OBLIGATION.


In [ ]:
# ━━━ Phase 1.1 — Synthetic Demo Data Generation ━━━
import copy

DEMO_RECORDS = [
    {
        "metadata": {"source_id": "REG-001", "regulation_name": "GDPR", "section": "Art 32"},
        "instruction": "Analyze the provided regulatory text to extract specific compliance obligations.",
        "context": "The controller shall implement appropriate technical measures.",
        "tool_use": {
            "available_tools": ["lookup_definition"],
            "tool_rationale": "'Technical measures' requires clarification.",
            "tool_call": {"function": "lookup_definition", "parameters": {"term": "technical measures"}},
            "external_context": "ISO27001-aligned controls."
        },
        "thought_trace": "1. Identify Subject: controller. 2. Identify Action: implement. 3. Identify Modality: MUST. 4. Conclusion: Obligation found.",
        "output": [{
            "obligation_id": "OBL-001", "subject": "controller", "modality": "MUST",
            "action": "implement appropriate technical measures", "reference_anchor": "Art 32"
        }]
    },
    {
        "metadata": {"source_id": "REG-002", "regulation_name": "General policy", "section": "Intro"},
        "instruction": "Analyze the provided regulatory text to extract specific compliance obligations.",
        "context": "The user may choose to log in via SSO.",
        "thought_trace": "1. Identify verb. 2. Evaluate modal 'may'. 3. Conclusion: Optional.",
        "output": {"status": "rejected", "message": "No obligation present", "reason": "Text uses permissive 'may'."}
    }
]

# Generate a fallback pool of synthetic data
all_synthetic_records = []
for i in range(100):
    for idx, rec in enumerate(DEMO_RECORDS):
        r = copy.deepcopy(rec)
        r["id"] = f"demo_{idx}_copy{i}"
        all_synthetic_records.append(r)

print(f'  Generated {len(all_synthetic_records)} synthetic samples.')

  Generated 200 synthetic samples.
  Records loaded: 860
✅ Valid: 0  ❌ Invalid: 860


In [6]:
# ━━━ Phase 1.2 — Data Ingestion, Mapping & Validation ━━━
def load_and_map_jsonl(path: str) -> List[dict]:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                rec = json.loads(line)
                # Map legacy 'source_text' or 'input_text' to 'context'
                if 'source_text' in rec and 'context' not in rec: rec['context'] = rec.pop('source_text')
                elif 'input_text' in rec and 'context' not in rec: rec['context'] = rec.pop('input_text')

                # Fix: Ensure context and instruction are not None (schema requirement)
                if rec.get('context') is None: rec['context'] = ""
                if rec.get('instruction') is None: rec['instruction'] = "Analyze the regulatory text."

                # Map legacy 'obligations' to 'output'
                if 'obligations' in rec and 'output' not in rec:
                    mapped_obs = []
                    for ob in rec['obligations']:
                        if 'obligation_text' in ob: ob['action'] = ob.pop('obligation_text')
                        if 'reference_anchor' not in ob: ob['reference_anchor'] = "N/A"
                        if 'obligation_id' not in ob: ob['obligation_id'] = f"OBL-{random.randint(1000,9999)}"
                        # Ensure string fields are not None
                        for field in ['subject', 'action', 'conditions', 'deadline']:
                            if ob.get(field) is None and field in ob: ob[field] = ""
                        mapped_obs.append(ob)
                    rec['output'] = mapped_obs

                # Fix: Map invalid status values to 'rejected'
                if isinstance(rec.get('output'), dict) and 'status' in rec['output']:
                    if rec['output']['status'] not in ['rejected', 'no_obligation']:
                        rec['output']['status'] = 'rejected'

                records.append(rec)
            except Exception as e: print(f'☀ Line {i}: Mapping error {e}')
    return records

# Execution Flow
if Path(cfg['data_path']).exists():
    raw_records = load_and_map_jsonl(cfg['data_path'])
    print(f'📂 Loaded {len(raw_records)} records from {cfg['data_path']}')
else:
    raw_records = all_synthetic_records
    print(f' ℹ️  Using synthetic demo records.')

random.shuffle(raw_records)

# Validate
valid_records = []
errors = []
for r in raw_records:
    ok, msg = validate_record(r)
    if ok: valid_records.append(r)
    else: errors.append(msg)

print(f'✅ Valid: {len(valid_records)}  ❌ Invalid: {len(raw_records) - len(valid_records)}')
if errors: print(f'☕ First Validation Error: {errors[0]}')

# Data Splitting
def split_dataset(records, train=0.8, val=0.1, test=0.1, seed=42):
    train_recs, temp = train_test_split(records, test_size=(val + test), random_state=seed)
    val_recs, test_recs = train_test_split(temp, test_size=test / (val + test), random_state=seed)
    return train_recs, val_recs, test_recs

if valid_records:
    train_recs, val_recs, test_recs = split_dataset(valid_records)
    print(f'━━━ Split Summary: Train={len(train_recs)} | Val={len(val_recs)} | Test={len(test_recs)}')
else:
    print("☠ No valid records to split. Check the validation errors above.")

📂 Loaded 860 records from resampled_ft-v2.jsonl
✅ Valid: 483  ❌ Invalid: 377
☕ First Validation Error: None is not of type 'string'
━━━ Split Summary: Train=386 | Val=48 | Test=49


In [7]:
# ━━━ Diagnostic: Why are records failing? ━━━
if 'errors' in globals() and errors:
    import pandas as pd
    error_df = pd.DataFrame({'error_message': errors})
    print('☁  Validation Failure Summary:')
    display(error_df['error_message'].value_counts().to_frame('count'))
else:
    print('✅ No invalid records found in the current run.')

☁  Validation Failure Summary:


,count
error_message,
None is not of type 'string',377


In [8]:
import copy
from typing import Tuple

def clean_none_values(obj, key_name=None):
    """Recursively convert None values to empty strings, except for specific schema keys that require null."""
    null_mandatory_fields = ['tool_use', 'thought_trace', 'tool_call', 'available_tools', 'tool_rationale', 'external_context']

    if key_name in null_mandatory_fields and obj is None:
        return None

    if isinstance(obj, dict):
        return {k: clean_none_values(v, k) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_none_values(i) for i in obj]
    elif obj is None:
        return ""
    return obj

def validate_record_permissive(record: dict) -> Tuple[bool, str]:
    """An even more relaxed validator that fixes nested schema violations and modality formatting."""
    rec = clean_none_values(copy.deepcopy(record))

    # 1. Schema specific fix: These cannot be empty strings, must be object or null
    for field in ['tool_use', 'thought_trace']:
        if rec.get(field) == "":
            rec[field] = None

    # 2. Fix Modality formatting (e.g., MUST NOT -> MUST_NOT for enum compatibility)
    if isinstance(rec.get('output'), list):
        for item in rec['output']:
            if isinstance(item, dict) and 'modality' in item:
                val = str(item['modality']).upper().replace(" ", "_")
                item['modality'] = val

    # 3. Ensure metadata exists with defaults if missing
    if not rec.get('metadata'):
        rec['metadata'] = {"source_id": "UNKNOWN", "regulation_name": "UNKNOWN", "section": "UNKNOWN"}
    else:
        for k in ["source_id", "regulation_name", "section"]:
            if k not in rec['metadata'] or not rec['metadata'][k]:
                rec['metadata'][k] = "UNKNOWN"

    # 4. Handle 'output' status mapping
    if not rec.get('output'):
        rec['output'] = {"status": "no_obligation", "message": "Auto-filled", "reason": "Missing output field"}
    elif isinstance(rec['output'], dict):
        status_map = {"no_content": "no_obligation", "incomplete": "rejected", "": "no_obligation"}
        current_status = rec['output'].get('status', "")
        if current_status in status_map or current_status not in ["rejected", "no_obligation"]:
            rec['output']['status'] = status_map.get(current_status, "rejected")

    try:
        validate(instance=rec, schema=OBLIGATION_SCHEMA)
        return True, ""
    except ValidationError as e:
        return False, e.message

# Re-run validation
valid_records = []
errors = []

for r in raw_records:
    ok, msg = validate_record_permissive(r)
    if ok:
        valid_records.append(r)
    else:
        errors.append(msg)

print(f'✅ Valid: {len(valid_records)}  ❌ Still Invalid: {len(raw_records) - len(valid_records)}')
if len(valid_records) > 0:
    train_recs, val_recs, test_recs = split_dataset(valid_records)
    # ─── 80 / 10 / 10 Split ──────────────────────────────────────────────────────
    print(f'━━━ Re-Split Summary: Train={len(train_recs)} | Val={len(val_recs)} | Test={len(test_recs)}')
if errors:
    print(f'Remaining Error Example: {errors[0]}')

✅ Valid: 860  ❌ Still Invalid: 0
━━━ Re-Split Summary: Train=688 | Val=86 | Test=86


## 🤖 Phase 2 — Load Model in 4-bit (QLoRA)

> **How QLoRA works:** The base model weights are quantized to 4-bit NF4 format using double quantization. LoRA adapters are added in full precision (FP16/BF16) on top. During the forward pass, weights are dequantized on-the-fly for computation, then discarded. This reduces base model VRAM from ~4.9 GB (BF16) to ~1.7 GB (4-bit), enabling fine-tuning within T4's 16 GB budget.

In [9]:
# ─── Aggressive Cleanup & Fragmentation Fix ━━━━━━━━━━━━━━━━━━━━━━━
import os, gc, torch

# 1. Set environment variable to reduce fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 2. Clear variables
for var in ['model', 'tokenizer', 'trainer']:
    if var in globals(): del globals()[var]
    if var in locals(): del locals()[var]

# 3. Hard reset of memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

# 4. Load using Unsloth's optimized 4-bit model directly
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = cfg["model_name"],
    max_seq_length = cfg["max_seq_length"],
    dtype        = cfg["dtype"],
    load_in_4bit = True,
)

print(f'✅ Model loaded using pre-quantized weights to conserve VRAM.')

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

✅ Model loaded using pre-quantized weights to conserve VRAM.


In [10]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
import json

SYSTEM_PROMPT = """You are a legal NLP expert specializing in regulatory obligation extraction.

Your task is to analyze the provided regulatory text and extract all obligations.
For each extraction, you must follow a strict 'thought_trace' to identify the subject, action, and modality (MUST, SHOULD, MAY, PROHIBITED, MUST_NOT, SHOULD_NOT, ASPIRATIONAL_OBLIGATION).

If the terminology is ambiguous, specify a 'tool_rationale'.

Output MUST be a valid JSON object matching the defined schema."""

def format_prompt(record: dict, include_answer: bool = True) -> str:
    instruction = record.get('instruction') or "Analyze the provided regulatory text."
    context = str(record.get('context', ''))
    metadata_str = json.dumps(record.get('metadata', {}))

    user_msg = f"Instruction: {instruction}\nContext: {context}\nMetadata: {metadata_str}"

    if include_answer:
        ans_data = {
            "tool_use": record.get("tool_use"),
            "thought_trace": record.get("thought_trace") or "Analyze context.",
            "output": record.get("output")
        }
        return f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{json.dumps(ans_data)}<end_of_turn>"
    return f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n"

def tokenize_fn(examples):
    texts = [str(t) if t is not None else "" for t in examples["text"]]
    if tokenizer is None:
        raise ValueError("Tokenizer is not initialized.")
    # Fix: Use explicit 'text' keyword argument for Gemma 4 processor compatibility
    return tokenizer(text=texts, truncation=True, max_length=cfg["max_seq_length"], padding="max_length")

def records_to_dataset(records):
    prompts = [{"text": format_prompt(r)} for r in records]
    ds = Dataset.from_list(prompts)
    return ds.map(tokenize_fn, batched=True, remove_columns=["text"])

train_ds = records_to_dataset(train_recs)
val_ds   = records_to_dataset(val_recs)
test_ds  = records_to_dataset(test_recs)

print(f'✅ Datasets tokenized: Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)}')

Map:   0%|          | 0/688 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

✅ Datasets tokenized: Train=688 | Val=86 | Test=86


In [11]:
# ─── Apply LoRA on 4-bit base (QLoRA) ━━━━━━━━━━━━━━━━━━━━
# Note: The 'model' was already loaded with load_in_4bit=True in cell j_IAgVa0BrIz
model = FastLanguageModel.get_peft_model(
    model,
    r                   = cfg["lora_r"],
    target_modules      = cfg["lora_target_modules"],
    lora_alpha          = cfg["lora_alpha"],
    lora_dropout        = cfg["lora_dropout"],
    bias                = "none",
    use_gradient_checkpointing = "unsloth", # Required for memory efficiency in QLoRA
    random_state        = 42,
    use_rslora          = False,
    loftq_config        = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ QLoRA adapters (4-bit + LoRA) attached to {cfg["model_name"]}.')
print(f'   Trainable params : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   Frozen params    : {total - trainable:,}')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ QLoRA adapters (4-bit + LoRA) attached to google/gemma-4-E2B-it.
   Trainable params : 9,289,728 (0.21%)
   Frozen params    : 4,408,311,328


## 📊 Phase 2.1 — MLflow Setup & Drift Monitoring

In [12]:
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment(cfg["mlflow_experiment"])

# ─── Data Drift Monitor ───────────────────────────────────────────────────────────
class DataDriftMonitor:
    def __init__(self, reference_records: List[dict]):
        self.ref_stats = self._compute_stats(reference_records)

    def _compute_stats(self, records: List[dict]) -> dict:
        # Updated keys: 'source_text' -> 'context', 'obligations' -> 'output'
        lengths   = [len(r.get('context', '')) for r in records]

        # Process 'output' based on whether it is a list (valid obligations) or dict (rejected)
        ob_counts = []
        mods      = []
        for r in records:
            out = r.get('output', [])
            if isinstance(out, list):
                ob_counts.append(len(out))
                mods.extend([ob.get('modality', 'UNKNOWN') for ob in out])
            else:
                ob_counts.append(0)

        return {
            'mean_text_len': np.mean(lengths),
            'std_text_len':  np.std(lengths),
            'mean_ob_count': np.mean(ob_counts),
            'modality_dist': pd.Series(mods).value_counts(normalize=True).to_dict() if mods else {}
        }

    def compute_drift(self, current_records: List[dict]) -> dict:
        cur = self._compute_stats(current_records)
        drift = {
            'text_len_drift': abs(cur['mean_text_len'] - self.ref_stats['mean_text_len']) / (self.ref_stats['mean_text_len'] + 1e-8),
            'ob_count_drift': abs(cur['mean_ob_count'] - self.ref_stats['mean_ob_count']) / (self.ref_stats['mean_ob_count'] + 1e-8),
        }
        all_keys = set(self.ref_stats['modality_dist']) | set(cur['modality_dist'])
        drift['modality_dist_l1'] = sum(
            abs(self.ref_stats['modality_dist'].get(k, 0) - cur['modality_dist'].get(k, 0))
            for k in all_keys
        )
        return drift

# ─── Enhanced MLflow Callback with QLoRA-specific VRAM tracking ───────────────
class MLflowCallback(TrainerCallback):

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            step_metrics = {k: v for k, v in logs.items() if isinstance(v, (int, float))}
            if torch.cuda.is_available():
                step_metrics['gpu_vram_used_gb']  = torch.cuda.memory_allocated() / 1e9
                step_metrics['gpu_vram_peak_gb']  = torch.cuda.max_memory_allocated() / 1e9
                step_metrics['gpu_vram_free_gb']  = (
                    torch.cuda.get_device_properties(0).total_memory -
                    torch.cuda.memory_allocated()
                ) / 1e9
            mlflow.log_metrics(step_metrics, step=state.global_step)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            mlflow.log_metrics(
                {k: v for k, v in metrics.items() if isinstance(v, (int, float))},
                step=state.global_step
            )

    def on_epoch_end(self, args, state, control, **kwargs):
        """Log per-epoch snapshot for drift monitoring."""
        if torch.cuda.is_available():
            mlflow.log_metrics({
                "epoch_vram_peak_gb": torch.cuda.max_memory_allocated() / 1e9
            }, step=state.epoch)
            torch.cuda.reset_peak_memory_stats()

drift_monitor = DataDriftMonitor(train_recs)
val_drift = drift_monitor.compute_drift(val_recs)
print('቉ Data drift (val vs train):', {k: f"{v:.4f}" for k, v in val_drift.items()})
print('✅ MLflow + drift monitor ready.')

቉ Data drift (val vs train): {'text_len_drift': '0.1281', 'ob_count_drift': '0.1739', 'modality_dist_l1': '0.1975'}
✅ MLflow + drift monitor ready.


## 🏋️ Phase 2.2 — Fine-Tuning with SFTTrainer (QLoRA)

In [13]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir                  = cfg["output_dir"],
    num_train_epochs            = cfg["num_train_epochs"],
    per_device_train_batch_size = cfg["per_device_train_batch_size"],
    gradient_accumulation_steps = cfg["gradient_accumulation_steps"],
    learning_rate               = cfg["learning_rate"],
    fp16                        = True,
    bf16                        = False,
    optim                       = "adamw_8bit",
    logging_steps               = cfg["logging_steps"],
    eval_strategy               = "steps",
    eval_steps                  = cfg["eval_steps"],
    save_strategy               = "steps",
    save_steps                  = cfg["save_steps"],
    report_to                   = "mlflow",
    max_seq_length              = cfg["max_seq_length"],
    dataset_text_field          = None,
    # Fix: Set remove_unused_columns=True since our dataset is already tokenized
    remove_unused_columns       = True,
    dataset_kwargs              = {"skip_prepare_dataset": True},
)

# Re-initialize the trainer with updated config
trainer = SFTTrainer(
    model           = model,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    args            = sft_config,
    processing_class = tokenizer,
)

print('✅ SFTTrainer re-initialized with remove_unused_columns=True for tokenized datasets.')

✅ SFTTrainer re-initialized with remove_unused_columns=True for tokenized datasets.


In [ ]:
# ─── Training Run with MLflow Tracking ━━━━━━━━━━━━━━━━━━━━━
import mlflow
from trl import SFTTrainer

# Ensure we are in a fresh MLflow run to avoid parameter conflicts
if mlflow.active_run():
    mlflow.end_run()

# Re-initialize the trainer right before training to ensure it is in memory
trainer = SFTTrainer(
    model           = model,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    args            = sft_config,
    processing_class = tokenizer,
)

# Workaround for TRL label handling
trainer.label_names = ["labels"]

with mlflow.start_run(run_name=cfg["run_name"]) as run:
    # Log config params
    params_to_log = {
        "base_model_path":   cfg["model_name"],
        "lora_r":            cfg["lora_r"],
        "lora_alpha":        cfg["lora_alpha"],
        "learning_rate":     cfg["learning_rate"],
        "epochs":            cfg["num_train_epochs"],
        "batch_size":        cfg["per_device_train_batch_size"],
        "grad_accum":        cfg["gradient_accumulation_steps"],
        "technique":         "QLoRA",
        "quantization":      "4-bit NF4",
    }

    for k, v in params_to_log.items():
        mlflow.log_param(k, v)

    # Log data drift
    mlflow.log_metrics({f"data_drift/{k}": v for k, v in val_drift.items()})

    print('• Starting training...')
    train_result = trainer.train()

    # Log final train metrics
    mlflow.log_metrics({
        "final_train_loss":      train_result.training_loss,
        "total_steps":           train_result.global_step,
    })

    # Save adapter weights
    model.save_pretrained(cfg["output_dir"])
    tokenizer.save_pretrained(cfg["output_dir"])
    mlflow.log_artifacts(cfg["output_dir"], artifact_path="lora_adapters")

    print(f'✅ Training complete. Run ID: {run.info.run_id}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


• Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 688 | Num Epochs = 5 | Total steps = 860
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,289,728 of 5,132,467,744 (0.18% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.588630,4.468343
100,0.702227,3.924542
150,0.374697,3.902121
200,0.332053,3.870579
250,0.239711,3.793985


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in ./gemma4_qlora_regulatory/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./gemma4_qlora_regulatory/checkpoint-200/tokenizer_config.json.


## 📏 Phase 3 — Evaluation: Span-F1 & Modality Accuracy

In [ ]:
FastLanguageModel.for_inference(model)

def extract_json_from_output(text: str) -> Optional[List[dict]]:
    match = re.search(r'(\[.*\])', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except json.JSONDecodeError: pass
    return None

def predict(record: dict, max_new_tokens: int = 512) -> Optional[List[dict]]:
    prompt = format_prompt(record, include_answer=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return extract_json_from_output(decoded)

print('✅ Inference helper ready.')

In [ ]:
def span_iou(ps, pe, gs, ge) -> float:
    i_s, i_e = max(ps, gs), min(pe, ge)
    if i_e <= i_s: return 0.0
    inter = i_e - i_s
    union = (pe - ps) + (ge - gs) - inter
    return inter / union if union > 0 else 0.0

def compute_span_f1(predictions, references, iou_threshold=0.5):
    tp = fp = fn = 0
    for preds, golds in zip(predictions, references):
        preds, golds = preds or [], golds or []
        matched = set()
        for p in preds:
            best_iou, best_j = 0.0, -1
            for j, g in enumerate(golds):
                if j in matched: continue
                iou = span_iou(p.get('span_start',0), p.get('span_end',0),
                               g.get('span_start',0), g.get('span_end',0))
                if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= iou_threshold: tp += 1; matched.add(best_j)
            else: fp += 1
        fn += len(golds) - len(matched)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1        = 2*precision*recall/(precision+recall) if precision+recall > 0 else 0.0
    return {"span_precision": precision, "span_recall": recall, "span_f1": f1,
            "tp": tp, "fp": fp, "fn": fn}

def compute_modality_accuracy(predictions, references, iou_threshold=0.5):
    all_pred, all_gold = [], []
    for preds, golds in zip(predictions, references):
        preds, golds = preds or [], golds or []
        matched = set()
        for p in preds:
            best_iou, best_j = 0.0, -1
            for j, g in enumerate(golds):
                if j in matched: continue
                iou = span_iou(p.get('span_start',0), p.get('span_end',0),
                               g.get('span_start',0), g.get('span_end',0))
                if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= iou_threshold and best_j >= 0:
                all_pred.append(p.get('modality','UNKNOWN'))
                all_gold.append(golds[best_j].get('modality','UNKNOWN'))
                matched.add(best_j)
    if not all_pred: return {"modality_accuracy": 0.0}
    acc = sum(p==g for p,g in zip(all_pred,all_gold)) / len(all_pred)
    rpt = classification_report(all_gold, all_pred,
                                 labels=["MUST","SHOULD","MAY","MUST_NOT","SHOULD_NOT"],
                                 zero_division=0, output_dict=True)
    return {"modality_accuracy": acc, "modality_report": rpt}

print('✅ Evaluation metrics defined.')

In [ ]:
print(f'🔍 Evaluating on {len(test_recs)} test samples...')
all_preds, all_golds, failed = [], [], 0

for i, record in enumerate(test_recs):
    pred_obs = predict(record)
    if pred_obs is None: pred_obs = []; failed += 1
    all_preds.append(pred_obs)
    all_golds.append(record['obligations'])
    if (i+1) % 10 == 0:
        print(f'  [{i+1}/{len(test_recs)}] parse failures: {failed}')

span_metrics     = compute_span_f1(all_preds, all_golds)
modality_metrics = compute_modality_accuracy(all_preds, all_golds)

print('\n📊 Span-Level Metrics (QLoRA):')
print(f"  Precision : {span_metrics['span_precision']:.4f}")
print(f"  Recall    : {span_metrics['span_recall']:.4f}")
print(f"  F1        : {span_metrics['span_f1']:.4f}")
print(f"\n📊 Modality Accuracy : {modality_metrics['modality_accuracy']:.4f}")
print(f"   JSON Parse Failures: {failed}/{len(test_recs)}")

In [ ]:
eval_metrics = {
    "test/span_precision":     span_metrics['span_precision'],
    "test/span_recall":        span_metrics['span_recall'],
    "test/span_f1":            span_metrics['span_f1'],
    "test/modality_accuracy":  modality_metrics['modality_accuracy'],
    "test/json_parse_failure": failed / max(len(test_recs), 1),
}

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metrics(eval_metrics)
    if 'modality_report' in modality_metrics:
        rpt_path = os.path.join(cfg.output_dir, "qlora_modality_report.json")
        with open(rpt_path, 'w') as f:
            json.dump(modality_metrics['modality_report'], f, indent=2)
        mlflow.log_artifact(rpt_path)

print('✅ Evaluation metrics logged to MLflow.')

## 🏛️ Phase 3.1 — Benchmarking: LegalBench & ObliQA

In [ ]:
def run_legalbench_eval(model, tokenizer, subset="contract_nli", max_samples=50):
    try:
        ds = load_dataset("nguha/legalbench", subset, split="test", trust_remote_code=True)
    except Exception as e:
        print(f'⚠️ LegalBench "{subset}" unavailable: {e}')
        return {"legalbench_accuracy": None}

    samples = ds.shuffle(seed=42).select(range(min(max_samples, len(ds))))
    preds, golds = [], []

    for sample in samples:
        prompt = (
            f"<start_of_turn>system\nYou are a legal expert. Answer only with the label.<end_of_turn>\n"
            f"<start_of_turn>user\n{sample['text']}\nAnswer:<end_of_turn>\n<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=16, temperature=0.1,
                                  do_sample=True, pad_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True).strip().upper()
        preds.append(decoded[:20])
        golds.append(str(sample.get('answer', sample.get('label', ''))).upper())

    acc = sum(p==g for p,g in zip(preds,golds)) / len(golds) if golds else 0
    return {"legalbench_accuracy": acc, "subset": subset, "n_samples": len(golds)}


def run_obliqa_eval(model, tokenizer, max_samples=50):
    try:
        ds = load_dataset("rcraigfieldwork/ObliQA", split="test", trust_remote_code=True)
        obliqa_samples = [{"context": s.get('passage', s.get('context','')),
                           "question": s['question'],
                           "answer": s.get('answer', s.get('answers',[''])[0])}
                          for s in ds.shuffle(seed=42).select(range(min(max_samples, len(ds))))]
    except Exception as e:
        print(f'⚠️ ObliQA unavailable: {e}. Using synthetic samples.')
        obliqa_samples = [
            {"context": "The data controller must obtain explicit consent before processing special category data.",
             "question": "What must the data controller obtain before processing special category data?",
             "answer": "explicit consent"},
            {"context": "Financial institutions should submit quarterly reports to the regulator within 30 days.",
             "question": "When should financial institutions submit quarterly reports?",
             "answer": "within 30 days"},
        ]

    rouge = evaluate.load("rouge")
    preds, golds = [], []

    for s in obliqa_samples:
        prompt = (
            f"<start_of_turn>system\nAnswer the question based on the regulatory context. Be concise.<end_of_turn>\n"
            f"<start_of_turn>user\nContext: {s['context']}\nQuestion: {s['question']}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, temperature=0.1,
                                  do_sample=True, pad_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        preds.append(decoded)
        golds.append(s['answer'])

    em = sum(p.lower().strip()==g.lower().strip() for p,g in zip(preds,golds)) / len(golds)
    rouge_scores = rouge.compute(predictions=preds, references=golds)
    return {"obliqa_exact_match": em, "obliqa_rouge1": rouge_scores["rouge1"],
            "obliqa_rougeL": rouge_scores["rougeL"], "n_samples": len(golds)}


print('Running LegalBench...')
lb_results = run_legalbench_eval(model, tokenizer, subset="contract_nli", max_samples=50)
print(f'LegalBench accuracy: {lb_results.get("legalbench_accuracy")}')

print('Running ObliQA...')
obliqa_results = run_obliqa_eval(model, tokenizer, max_samples=50)
print(f'ObliQA Exact Match: {obliqa_results["obliqa_exact_match"]:.4f}')
print(f'ObliQA ROUGE-L    : {obliqa_results["obliqa_rougeL"]:.4f}')

In [ ]:
benchmark_metrics = {
    "benchmark/legalbench_accuracy":  lb_results.get('legalbench_accuracy') or 0.0,
    "benchmark/obliqa_exact_match":   obliqa_results['obliqa_exact_match'],
    "benchmark/obliqa_rouge1":        obliqa_results['obliqa_rouge1'],
    "benchmark/obliqa_rougeL":        obliqa_results['obliqa_rougeL'],
}

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metrics(benchmark_metrics)

print('\n📊 Final QLoRA Benchmark Summary')
print('=' * 44)
for k, v in {**eval_metrics, **benchmark_metrics}.items():
    print(f'  {k:<42} {v:.4f}')

## 🔍 Phase 4 — Model Drift Monitoring

In [ ]:
def compute_prediction_distribution(predictions):
    all_mods = [ob.get('modality','UNKNOWN') for preds in predictions for ob in (preds or [])]
    total = len(all_mods)
    return pd.Series(all_mods).value_counts(normalize=True).to_dict() if total else {}

pred_dist = compute_prediction_distribution(all_preds)
gold_dist = compute_prediction_distribution(all_golds)

print('📊 QLoRA Prediction vs Gold Modality Distribution:')
print(f'{"Modality":<15} {"Predicted":>12} {"Gold":>12}')
print('-' * 40)
for mod in sorted(set(pred_dist) | set(gold_dist)):
    print(f'{mod:<15} {pred_dist.get(mod,0):>12.3f} {gold_dist.get(mod,0):>12.3f}')

model_drift_l1 = sum(abs(pred_dist.get(k,0) - gold_dist.get(k,0))
                      for k in set(pred_dist)|set(gold_dist))
print(f'\n🔀 Model drift (L1 divergence): {model_drift_l1:.4f}')
print('   Threshold alert ⚠️' if model_drift_l1 > 0.3 else '   ✅ Within acceptable range')

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metric("model_drift/modality_l1", model_drift_l1)

## 📊 Phase 4.1 — LoRA vs QLoRA Comparison Table

Use this cell after running both notebooks to compare results.

In [ ]:
# Fill in LoRA results from the other notebook for comparison
comparison = {
    "Metric":              ["Span Precision", "Span Recall", "Span F1",
                            "Modality Accuracy", "LegalBench Acc",
                            "ObliQA EM", "ObliQA ROUGE-L",
                            "VRAM Peak (GB)", "Train Time (min)"],
    "LoRA (BF16)": ["—"] * 9,   # ← paste from LoRA notebook
    "QLoRA (NF4)": [
        f"{span_metrics['span_precision']:.4f}",
        f"{span_metrics['span_recall']:.4f}",
        f"{span_metrics['span_f1']:.4f}",
        f"{modality_metrics['modality_accuracy']:.4f}",
        f"{lb_results.get('legalbench_accuracy', 0):.4f}",
        f"{obliqa_results['obliqa_exact_match']:.4f}",
        f"{obliqa_results['obliqa_rougeL']:.4f}",
        f"{torch.cuda.max_memory_allocated()/1e9:.2f}" if torch.cuda.is_available() else "—",
        f"{train_result.metrics.get('train_runtime', 0)/60:.1f}"
    ]
}

df_cmp = pd.DataFrame(comparison)
print(df_cmp.to_string(index=False))

# Save comparison table
cmp_path = os.path.join(cfg.output_dir, "lora_vs_qlora_comparison.csv")
df_cmp.to_csv(cmp_path, index=False)
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_artifact(cmp_path)
print(f'✅ Comparison table saved to {cmp_path}')

## 💾 Phase 5 — Save & Export

In [ ]:
# ─── Save QLoRA Adapters (always lightweight — base model is frozen) ───────────
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f'✅ QLoRA adapters saved to {cfg.output_dir}')

# ─── Merge into BF16 for deployment (optional) ────────────────────────────────
MERGE_AND_SAVE = False  # Set True to merge — note: merges to BF16 (not 4-bit)
if MERGE_AND_SAVE:
    merged_dir = cfg.output_dir + "_merged_bf16"
    print(f'Merging QLoRA adapters into BF16 base → {merged_dir}')
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print('✅ Merged BF16 model saved.')

# ─── Save as GGUF for llama.cpp / Ollama (optional) ──────────────────────────
SAVE_GGUF = False
if SAVE_GGUF:
    model.save_pretrained_gguf(cfg.output_dir, tokenizer, quantization_method="q4_k_m")
    print('✅ GGUF Q4_K_M model saved.')

# ─── Push to Hub ──────────────────────────────────────────────────────────────
PUSH_TO_HUB = False
HF_REPO     = "your-username/gemma4-qlora-regulatory"
if PUSH_TO_HUB:
    from huggingface_hub import login
    login(token="hf_...")   # replace
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)

print('\n🎉 QLoRA Pipeline complete!')
print(f'   Output dir     : {cfg.output_dir}')
print(f'   MLflow run ID  : {run.info.run_id}')
print(f'   Span F1        : {span_metrics["span_f1"]:.4f}')
print(f'   Modality Acc   : {modality_metrics["modality_accuracy"]:.4f}')

In [ ]:
# ─── MLflow UI ────────────────────────────────────────────────────────────────
import subprocess, threading

def run_mlflow_ui():
    subprocess.run(["mlflow", "ui", "--port", "5000", "--host", "0.0.0.0"])

t = threading.Thread(target=run_mlflow_ui, daemon=True)
t.start()

try:
    from pyngrok import ngrok
    public_url = ngrok.connect(5000)
    print(f'🌐 MLflow UI: {public_url}')
except ImportError:
    print('MLflow running on port 5000. pip install pyngrok for a public URL.')